# 📚 第一阶段小结（004 - 014）

## 整体思维路线图

004-014 的学习内容可以分为 **三大板块**，它们层层递进，构建了深度学习的完整基础：

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                        第一阶段：深度学习基础                                  │
├─────────────┬──────────────────────────┬────────────────────────────────────┤
│  板块一      │       板块二              │          板块三                     │
│  数学工具箱   │     核心模型与训练         │        防止过拟合                   │
│ (004-007)   │      (008-010)           │        (011-014)                  │
├─────────────┼──────────────────────────┼────────────────────────────────────┤
│ 张量操作     │  线性回归(最简单的NN)      │  模型选择 / 过拟合 / 欠拟合          │
│ 线性代数     │  Softmax回归(分类)        │  权重衰退(L2正则)                   │
│ 矩阵求导     │  多层感知机(真正的NN)      │  丢弃法(Dropout)                   │
│ 自动求导     │  损失函数 / 优化算法       │  数值稳定性 / 权重初始化              │
└─────────────┴──────────────────────────┴────────────────────────────────────┘
```

**一句话总结学习脉络**：先学会操作数据和求导（工具），再学会搭建和训练模型（方法），最后学会让模型泛化得更好（技巧）。

---

# 🧮 板块一：数学工具箱（004 - 007）

> **目标**：掌握深度学习所需的数据结构、数学运算、求导方法，为后续搭建和训练模型打下基础。

---

## 004 数据操作与数据预处理

### 你学到了什么？

| 知识点 | 核心内容 | 关键API |
| :--- | :--- | :--- |
| **张量(Tensor)** | 深度学习中一切数据的载体，本质是多维数组 | `torch.arange`, `torch.zeros`, `reshape` |
| **广播机制** | 形状不同的张量运算时，自动扩展长度为1的维度 | 隐式触发，需检查`shape` |
| **原位操作** | 避免频繁内存分配，`Y[:] = X + Y` 而非 `Y = X + Y` | `Y += X`, `Y.add_(X)` |
| **数据预处理** | CSV读取 → 缺失值处理 → One-Hot编码 → 转Tensor | `pd.read_csv`, `pd.get_dummies` |
| **NumPy互转** | Tensor和NumPy共享内存，零拷贝互转 | `torch.from_numpy`, `.numpy()` |

### 核心思维
- 🔑 **广播是把双刃剑**：方便运算但可能隐藏shape错误，养成检查shape的习惯
- 🔑 **深度学习用float32**：Pandas默认float64，转Tensor时要指定`dtype=torch.float32`
- 🔑 **原位操作在Autograd中要谨慎**：被保存用于反向传播的张量不能原位修改

## 005 线性代数

### 你学到了什么？

| 知识点 | 核心内容 | 关键API |
| :--- | :--- | :--- |
| **标量/向量/矩阵/张量** | 0维→1维→2维→N维的逐步推广 | `torch.tensor`, `.reshape` |
| **矩阵运算** | 转置、哈达玛积(逐元素)、矩阵乘法(空间变换) | `.T`, `A*B`, `torch.mm(A,B)` |
| **降维求和** | 沿指定轴求和，`keepdim=True`保持维度便于广播 | `.sum(axis=0)`, `.mean()` |
| **向量点积** | 两向量对应元素相乘后求和 | `torch.dot(x,y)` |
| **矩阵向量积** | 矩阵乘向量 = 对向量做线性变换 | `torch.mv(A,x)` |
| **范数** | L1范数(绝对值之和)、L2范数(平方和开根)、F范数 | `torch.norm(u)`, `torch.abs(u).sum()` |

### 核心思维
- 🔑 **哈达玛积 vs 矩阵乘法**：前者是逐元素运算，后者是空间线性变换（神经网络的几何基石）
- 🔑 **L1产生稀疏性，L2让权重均匀变小**：L1在零点导数恒为±1，倾向于将权重压为0；L2导数随值减小而变小
- 🔑 **keepdim=True的必要性**：求和后保持维度才能与原张量广播运算

## 006 矩阵计算（求导法则）

### 你学到了什么？

| 知识点 | 核心内容 |
| :--- | :--- |
| **标量求导与链式法则** | 微积分基础，神经网络自动求导的数学支撑 |
| **亚导数(Subgradient)** | 不可导点（如`abs(x)`在x=0）的导数延伸，值为区间`[-1,1]` |
| **梯度的几何意义** | 梯度与等高线正交，指向函数值增长最快的方向 |
| **向量/矩阵求导布局** | 标量对向量、向量对向量(雅可比矩阵)、矩阵对矩阵 |
| **求导口诀** | **「上边不变，下边翻转」**，弄清输入输出维度关系 |

### 常用求导公式
$$\frac{\partial (\mathbf{a}^\top \mathbf{x})}{\partial \mathbf{x}} = \mathbf{a}^\top \quad\quad \frac{\partial (\mathbf{x}^\top \mathbf{A} \mathbf{x})}{\partial \mathbf{x}} = \mathbf{x}^\top(\mathbf{A} + \mathbf{A}^\top)$$

### 核心思维
- 🔑 **梯度指向上山方向**：所以优化时沿**负梯度**方向走（梯度下降）
- 🔑 **链式法则是反向传播的灵魂**：复合函数的导数 = 各层导数的连乘

## 007 自动求导（Autograd）

### 你学到了什么？

| 知识点 | 核心内容 |
| :--- | :--- |
| **计算图** | 将计算分解为节点（变量/算子）和有向边（依赖关系）的图结构 |
| **正向累积** | 从输入→输出依次计算导数，内存O(1)，但多输入时慢 |
| **反向累积(反向传播)** | 从输出→输入倒推导数，内存O(n)但在深度学习中极快 |
| **`requires_grad=True`** | 告诉PyTorch追踪该张量的所有运算 |
| **`backward()`** | 启动反向传播，计算梯度 |
| **`detach()`** | 将张量从计算图分离，视为常数（不参与梯度计算） |
| **`grad.zero_()`** | PyTorch梯度是累加的，每次迭代前需手动清零 |

### 核心思维
- 🔑 **为什么用反向传播而非正向**：深度学习 = 1个标量Loss + 百万参数，反向模式一次反传就能算出所有参数的梯度
- 🔑 **`detach()`的作用**：冻结某些参数、截断梯度流（如GAN训练中固定判别器）
- 🔑 **梯度会累加**：不清零会导致梯度越来越大，必须在每个batch开始前`zero_()`

---

# 🏗️ 板块二：核心模型与训练（008 - 010）

> **目标**：从最简单的线性模型出发，逐步构建到多层感知机，掌握「模型 + 损失 + 优化」的核心训练范式。

---

## 008 线性回归与优化算法

### 你学到了什么？

| 知识点 | 核心内容 |
| :--- | :--- |
| **线性回归** | $\mathbf{y} = \mathbf{X}\mathbf{w} + b + \epsilon$，最简单的单层神经网络 |
| **MSE损失** | 均方误差 $L = \frac{1}{n}\sum(y_i - \hat{y}_i)^2$，衡量预测与真实的偏差 |
| **解析解** | $\mathbf{w}^* = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$，但求逆复杂且无法推广到非线性 |
| **小批量SGD** | 每次用一小批数据估计梯度，平衡计算效率与梯度精度 |
| **学习率** | 步长太大震荡不收敛，太小收敛太慢 |
| **两种实现** | 从零实现（理解原理）→ 框架实现（工程效率） |

### 训练流程（深度学习的核心范式）
```
初始化参数 → for epoch → for batch:
    ① 前向传播：y_hat = net(X)
    ② 计算损失：loss = loss_fn(y_hat, y)
    ③ 反向传播：loss.backward()
    ④ 更新参数：optimizer.step()
    ⑤ 清零梯度：optimizer.zero_grad()
```

### 核心思维
- 🔑 **线性回归就是单层神经网络**：没有隐藏层，没有激活函数
- 🔑 **batch size是个权衡**：太小梯度噪声大但正则化效果好，太大需更多显存但梯度准确
- 🔑 **从零实现→框架实现**：先理解原理再用API，这个学习模式贯穿全课程

## 009 Softmax回归、损失函数与分类

### 你学到了什么？

| 知识点 | 核心内容 |
| :--- | :--- |
| **Softmax函数** | 将logits转为概率分布：$\text{softmax}(\mathbf{o})_i = \frac{\exp(o_i)}{\sum_k \exp(o_k)}$，所有输出和为1 |
| **交叉熵损失** | $\ell = -\log \hat{p}_y$，只关心正确类的预测概率，概率越大损失越小 |
| **三种损失函数** | L2(均方)、L1(绝对值)、Huber(两者结合) |
| **Fashion-MNIST** | 10类服装图片数据集，比MNIST更有挑战性 |
| **`nn.CrossEntropyLoss`** | 内部自动做log-softmax + NLLLoss，数值更稳定 |

### 三种损失函数对比

| 损失函数 | 公式 | 梯度特点 | 适用场景 |
| :--- | :--- | :--- | :--- |
| **L2 (MSE)** | $(y-\hat{y})^2$ | 远处梯度大，近处梯度小 | 一般回归 |
| **L1 (MAE)** | $|y-\hat{y}|$ | 梯度恒为±1，零点不可导 | 对异常值鲁棒 |
| **Huber** | 近处L2，远处L1 | 兼顾两者优点 | 鲁棒回归 |

### 核心思维
- 🔑 **回归 vs 分类**：回归预测连续值用MSE，分类预测概率用交叉熵
- 🔑 **不要在网络最后接Softmax再喂CrossEntropyLoss**：`nn.CrossEntropyLoss`已内置softmax
- 🔑 **最小化交叉熵 = 最大化正确类的对数似然**：两者数学等价

## 010 多层感知机（MLP）

### 你学到了什么？

| 知识点 | 核心内容 |
| :--- | :--- |
| **单层感知机** | 最简单的二分类线性模型，$y = \text{sign}(\mathbf{w}^\top \mathbf{x} + b)$ |
| **XOR问题** | 单层感知机无法解决非线性问题（只能画一条直线） |
| **多层感知机** | 加入隐藏层 + 非线性激活函数，解决非线性问题 |
| **为什么需要激活函数** | 没有激活函数，多少层线性层叠加都等价于一层线性层 |
| **激活函数** | ReLU / Sigmoid / Tanh |

### 三大激活函数

| 激活函数 | 公式 | 优点 | 缺点 |
| :--- | :--- | :--- | :--- |
| **ReLU** | $\max(0, x)$ | 计算简单、缓解梯度消失 | 负半轴梯度为0（Dead ReLU） |
| **Sigmoid** | $\frac{1}{1+e^{-x}}$ | 输出在(0,1)可做概率 | 梯度最大0.25、非零中心 |
| **Tanh** | $\frac{e^x-e^{-x}}{e^x+e^{-x}}$ | 零中心输出 | 饱和区梯度仍消失 |

### MLP网络结构
```
输入(784) → [Linear + ReLU] → 隐藏(256) → [Linear] → 输出(10)
              第1层(有参数)              第2层(有参数)

注：Flatten不算一层（无可学习参数），输出层一般不接ReLU
```

### 核心思维
- 🔑 **激活函数是深度学习的命门**：没有它，再深的网络也只是线性模型
- 🔑 **ReLU是默认选择**：计算快、不需要指数运算（CPU上一次exp ≈ 上百次乘法）
- 🔑 **MLP = 线性层 + 激活函数的堆叠**：这就是"深度"学习的起点

---

# 🛡️ 板块三：防止过拟合（011 - 014）

> **目标**：学会如何诊断和解决过拟合/欠拟合问题，让模型在新数据上表现更好。

---

## 011 模型选择、过拟合与欠拟合

### 你学到了什么？

| 知识点 | 核心内容 |
| :--- | :--- |
| **训练误差 vs 泛化误差** | 我们的目标是降低泛化误差，而非训练误差 |
| **欠拟合** | 训练误差和验证误差都高 → 模型太简单 |
| **过拟合** | 训练误差低但验证误差高 → 模型太复杂，学到了噪声 |
| **数据集划分** | 训练集(训练) / 验证集(调参) / 测试集(最终评估) |
| **K折交叉验证** | 数据不够时，将训练集分K份轮流做验证，取平均 |
| **模型容量** | 模型能拟合的函数的复杂度；容量太大容易过拟合 |

### 模型容量与误差的关系
```
误差 ↑
     │\                          ╱  泛化误差
     │ \                      ╱
     │  \                   ╱
     │   \      最优      ╱
     │    \     ↓       ╱
     │     \   •------╱
     │      \╱
     │       \____________  训练误差
     │
     └──────────────────────→ 模型容量
      欠拟合区    最佳区    过拟合区
```

### 核心思维
- 🔑 **测试集只能用一次**：如果用测试集调参，它就变成了验证集，泛化评估就失效了
- 🔑 **数据量 > 模型复杂度**：数据越多越不容易过拟合；数据少就要控制模型容量
- 🔑 **SVM vs 神经网络**：小数据+好特征用SVM；大数据+原始输入用深度学习

## 012 权重衰退（Weight Decay / L2正则化）

### 你学到了什么？

| 知识点 | 核心内容 |
| :--- | :--- |
| **核心思想** | 在损失函数中加入L2惩罚项：$L' = L + \frac{\lambda}{2}\|\mathbf{w}\|^2$ |
| **效果** | 惩罚大权重，鼓励权重分散且偏小，防止过拟合 |
| **参数更新** | $\mathbf{w} \leftarrow (1-\eta\lambda)\mathbf{w} - \eta\frac{\partial L}{\partial \mathbf{w}}$，每次更新先将权重缩小一点 |
| **超参数λ** | λ越大惩罚越重，权重越小；λ=0等于没有正则化 |

### 几何直觉
```
绿色等高线：原始损失L的最优解（模型可能过拟合的点）
橙色等高线：L2惩罚项（以原点为中心的圆）

加入惩罚后，最优解从原始最优点被「拉向原点」
→ 权重被限制在较小范围内
→ 模型不会过度依赖少数特征
```

### 核心思维
- 🔑 **损失函数 ≠ 目标函数**：目标函数 = 损失 + 正则项，最优解不同
- 🔑 **权重衰退的名字由来**：每次更新参数时先乘以 $(1-\eta\lambda)$，权重「衰退」了一点
- 🔑 **PyTorch实现**：`torch.optim.SGD(params, lr=lr, weight_decay=wd)`，一行搞定

## 013 丢弃法（Dropout）

### 你学到了什么？

| 知识点 | 核心内容 |
| :--- | :--- |
| **核心思想** | 训练时随机将隐藏层神经元以概率p清零，打破共适应性 |
| **期望无偏** | 存活的神经元除以(1-p)，保证 $E[h'_i] = h_i$ |
| **集成学习直觉** | 每次训练一个不同的子网络，测试时相当于所有子网络的集成 |
| **训练 vs 测试** | 训练：开启dropout；测试：关闭dropout（`model.eval()`） |

### Weight Decay vs Dropout 对比

| 维度 | 权重衰退 | 丢弃法 |
| :--- | :--- | :--- |
| **作用对象** | 权重参数 $\mathbf{w}$ | 激活值/层输出 $h$ |
| **机制** | 在目标函数加L2惩罚（确定性） | 随机清零输出（随机性） |
| **作用位置** | 全局控制（优化器参数） | 插在隐藏层之后 |
| **超参数** | $\lambda$（权重衰退率） | $p$（丢弃率，浅层~0.2，深层~0.5） |
| **能否联合使用** | ✅ 可以，且是标准做法 | ✅ 可以，且是标准做法 |

### 核心思维
- 🔑 **输出层不加Dropout**：输出层负责最终预测，丢弃会直接破坏预测
- 🔑 **为什么除以(1-p)**：保证训练和测试时输出的期望一致，测试时无需额外缩放
- 🔑 **Dropout = 穷人的集成学习**：$2^D$个子网络的隐式集成

## 014 数值稳定性、模型初始化与激活函数

### 你学到了什么？

| 知识点 | 核心内容 |
| :--- | :--- |
| **梯度爆炸** | 权重矩阵尺度>1，层数深时梯度指数级膨胀 → NaN/溢出 |
| **梯度消失** | 激活函数导数接近0（Sigmoid饱和区），梯度指数级衰减 → 浅层不更新 |
| **梯度裁剪** | 梯度范数超过阈值时按比例缩小，抑制梯度爆炸 |
| **Xavier初始化** | $\text{Var}(w) = \frac{2}{n_{in} + n_{out}}$，适用于Sigmoid/Tanh |
| **He/Kaiming初始化** | $\text{Var}(w) = \frac{2}{n_{in}}$，适用于ReLU（补偿一半清零） |
| **为什么不能全零初始化** | 所有神经元对称 → 梯度相同 → 退化为单神经元 |

### 数值不稳定的根因

反向传播中梯度涉及多层矩阵连乘：
$$\frac{\partial \mathcal{L}}{\partial z^{(l)}} = \frac{\partial \mathcal{L}}{\partial z^{(L)}} \prod_{k=l+1}^{L} W^{(k)} \text{diag}(\phi'(z^{(k-1)}))$$

- 连乘项 **> 1** → 梯度爆炸
- 连乘项 **< 1** → 梯度消失

### 初始化方法选择
```
激活函数是 Sigmoid / Tanh → Xavier初始化
激活函数是 ReLU / LeakyReLU → He/Kaiming初始化
```

### 核心思维
- 🔑 **Sigmoid的导数最大才0.25**：连乘n个<1的数，梯度消失是必然的
- 🔑 **ReLU缓解但不完全解决梯度消失**：正半轴导数恒为1，但负半轴为0（Dead ReLU）
- 🔑 **初始化 + BN是互补的**：初始化是训练前的静态方差控制，BN是训练中的动态归一化

---

# 🗺️ 总体知识图谱

```
数据操作(004)──→ 线性代数(005)──→ 矩阵求导(006)──→ 自动求导(007)
  │张量/广播        │向量/矩阵运算      │链式法则/梯度       │计算图/反向传播
  │                 │范数               │                  │
  └────────────────────────数学基础──────────────────────────┘
                                │
                                ▼
              线性回归(008)──→ Softmax回归(009)──→ 多层感知机(010)
                │单层NN           │分类任务             │隐藏层+激活函数
                │MSE损失          │交叉熵损失           │ReLU/Sigmoid/Tanh
                │小批量SGD        │Softmax函数          │
                │                 │                    │
                └──────────────模型与训练───────────────┘
                                │
                                ▼
        模型选择(011)──→ 权重衰退(012)──→ 丢弃法(013)──→ 数值稳定性(014)
          │过拟合/欠拟合     │L2正则化         │随机清零          │梯度爆炸/消失
          │验证集/K折CV     │惩罚大权重       │集成学习          │Xavier/He初始化
          │模型容量         │                │                 │激活函数选择
          │               │                │                 │
          └──────────────防止过拟合 & 训练稳定──────────────────┘
```

---

# 🎯 你现在应该能回答的关键问题

### 数学基础
1. 什么是广播机制？它有什么风险？
2. 矩阵乘法和哈达玛积有什么本质区别？
3. 梯度的几何意义是什么？它和等高线什么关系？
4. 为什么深度学习用反向传播而不是正向传播来求梯度？

### 模型与训练
5. 线性回归为什么可以看作单层神经网络？
6. Softmax函数的作用是什么？为什么分类用交叉熵而不用MSE？
7. 为什么需要激活函数？没有激活函数的多层网络等价于什么？
8. 深度学习训练的五步流程是什么？

### 防止过拟合
9. 如何判断模型是过拟合还是欠拟合？
10. 权重衰退和Dropout分别从什么角度防止过拟合？
11. 为什么不能把权重初始化为全零？
12. Xavier和He初始化分别适用于什么激活函数？为什么？

---

# ⏭️ 下一步展望

完成了第一阶段，你已经掌握了：
- ✅ 深度学习的数学语言（张量、求导、自动求导）
- ✅ 核心模型范式（线性回归 → Softmax → MLP）
- ✅ 训练技巧（正则化、初始化、数值稳定性）

接下来进入 **第二阶段**，你将学习：
- 📌 **015 Kaggle房价预测**：第一个实战项目，综合运用以上所有知识
- 📌 **016 PyTorch神经网络基础**：自定义层/块/参数管理
- 📌 **019+ 卷积神经网络**：从全连接到卷积，开启计算机视觉之旅

```
第一阶段(你在这里)    第二阶段             第三阶段           第四阶段
数学 + MLP基础   →  CNN卷积神经网络  →  RNN循环神经网络  →  注意力与Transformer
004-014            019-029            246-254            259-265
```